# Laboratorio de Big Data en Databricks

## Propósito del ejercicio
El objetivo principal de esta actividad es familiarizarse con Databricks y el manejo de datos masivos. Durante el laboratorio se realizarán las siguientes tareas:
- Cargar un conjunto de datos en Databricks.
- Explorar y comprender la estructura de los datos.
- Ejecutar consultas utilizando **SQL** y **PySpark**.
- Generar visualizaciones gráficas para analizar tendencias.
- Realizar un análisis básico de los resultados obtenidos.

Este ejercicio busca desarrollar habilidades prácticas en el uso de herramientas de Big Data.

## Fuente de datos
Se utilizarán datos de Market Data relacionados con criptomonedas, incluyendo:
- Bitcoin (BTC)
- Cardano (ADA)
- Otros activos digitales

Los datos contienen información como:
- Precio
- Volumen de transacciones
- Fecha y hora de registro

Estos datos permiten realizar análisis exploratorios sobre la evolución de los mercados financieros digitales.





##Paso 4. Exploración inicial de los datos

In [0]:
--Visualizacion primeras filas dataset

SELECT * FROM workspace.bigdata.coin_data_market
Limit 2;

coin,date,price,market_cap,total_volume,source,ingestion_date
bitcoin,2025-08-07,115013.39140866172,2.289094080155993E12,3.204811906660145E10,coingecko,2026-08-06
bitcoin,2025-08-08,117482.59621578844,2.3381089114879683E12,3.815164953563035E10,coingecko,2026-08-06


In [0]:

%python
#Creacion del dataframe y visualizacion de la estructura de la tabla
coin_data = spark.table("workspace.bigdata.coin_data_market")
display(coin_data)

coin,date,price,market_cap,total_volume,source,ingestion_date
bitcoin,2025-08-07,115013.39140866172,2.289094080155993E12,3.204811906660145E10,coingecko,2026-08-06
bitcoin,2025-08-08,117482.59621578844,2.3381089114879683E12,3.815164953563035E10,coingecko,2026-08-06
bitcoin,2025-08-09,116686.72473635004,2.322432134103607E12,3.309140747231744E10,coingecko,2026-08-06
bitcoin,2025-08-10,116500.75461167548,2.318845598641438E12,2.6325306630626644E10,coingecko,2026-08-06
bitcoin,2025-08-11,119296.65800359056,2.374034492454683E12,3.656288433413131E10,coingecko,2026-08-06
bitcoin,2025-08-12,118726.7609760874,2.3635434290220864E12,6.215812017553192E10,coingecko,2026-08-06
bitcoin,2025-08-13,120106.25587065212,2.3923442120790225E12,4.7014629997495224E10,coingecko,2026-08-06
bitcoin,2025-08-14,123419.33444694606,2.456741384606632E12,6.301757422332698E10,coingecko,2026-08-06
bitcoin,2025-08-15,118387.76940297017,2.3569017449332793E12,6.8707152125851295E10,coingecko,2026-08-06
bitcoin,2025-08-16,117396.51852350793,2.335290383260173E12,4.4195309908176605E10,coingecko,2026-08-06


In [0]:
%python
#Revision de identificacion de columnas, estructura general y conteo de registros.
coin_data.show(4)
coin_data.printSchema()
print(f"Total registros: {coin_data.count()}")

+-------+----------+------------------+--------------------+--------------------+---------+--------------+
|   coin|      date|             price|          market_cap|        total_volume|   source|ingestion_date|
+-------+----------+------------------+--------------------+--------------------+---------+--------------+
|bitcoin|2025-08-07|115013.39140866172|2.289094080155993E12|3.204811906660145E10|coingecko|    2026-08-06|
|bitcoin|2025-08-08|117482.59621578844|2.338108911487968...|3.815164953563035E10|coingecko|    2026-08-06|
|bitcoin|2025-08-09|116686.72473635004|2.322432134103607E12|3.309140747231744E10|coingecko|    2026-08-06|
|bitcoin|2025-08-10|116500.75461167548|2.318845598641438E12|2.632530663062664...|coingecko|    2026-08-06|
+-------+----------+------------------+--------------------+--------------------+---------+--------------+
only showing top 4 rows
root
 |-- coin: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- mar

In [0]:
%python
#Revision de valores nulos
from pyspark.sql import functions as F

coin_data.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in coin_data.columns
]).show()

+----+----+-----+----------+------------+------+--------------+
|coin|date|price|market_cap|total_volume|source|ingestion_date|
+----+----+-----+----------+------------+------+--------------+
|   0|   0|    0|         0|           0|     0|             0|
+----+----+-----+----------+------------+------+--------------+



##Los datos trabajados son datos estructurados

Características:
- Esquema rígido con filas y columnas.
- Tipos de datos definidos (texto, número, fecha).
- Fácil de consultar con SQL.


##Paso 5. Almacenamiento en formato tabular

###Ventajas de formato delta

Consultas rápidas: SQL directo sobre tablas
Consistencia ACID: Escrituras seguras en entornos distribuidos
Versionado: Recuperar estados anteriores del dataset
Escalabilidad: Manejo de grandes volúmenes de datos sin perder rendimiento

In [0]:
%python
#Tabla delta
coin_data.write.mode("overwrite").format("delta").saveAsTable("workspace.bigdata.coin_prices_market2")

In [0]:
DESCRIBE HISTORY workspace.bigdata.coin_prices_market2;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-08-20T00:42:33.000Z,71958481127059,daniicadenam@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2534627354347170),cc46f794-c53d-4f5b-8bf6-52f862d87536,0820-001650-ow4ugkj7-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1488, numOutputBytes -> 38309)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


##Paso 6. Consulta básica de datos


In [0]:
%python
#Numero total de registros
print(f"Numero total de registros:",{coin_data.count()}) 

Numero total de registros: {1488}


In [0]:
%sql
--Consulta columnas relevantes
SELECT coin, date, price
FROM workspace.bigdata.coin_data_market
LIMIT 5;

coin,date,price
bitcoin,2025-08-07,115013.39140866172
bitcoin,2025-08-08,117482.59621578844
bitcoin,2025-08-09,116686.72473635004
bitcoin,2025-08-10,116500.75461167548
bitcoin,2025-08-11,119296.65800359056


In [0]:
%sql
--Consulta con filtro -- Filtrar moneda bitcoin y precio mayor al promedio
SELECT coin, date, price
FROM workspace.bigdata.coin_data_market
WHERE coin = 'bitcoin' 
and price > (select avg(price) from workspace.bigdata.coin_data_market where coin = 'bitcoin')
ORDER BY date
LIMIT 5;

coin,date,price
bitcoin,2025-08-07,115013.39140866172
bitcoin,2025-08-08,117482.59621578844
bitcoin,2025-08-09,116686.72473635004
bitcoin,2025-08-10,116500.75461167548
bitcoin,2025-08-11,119296.65800359056


In [0]:
--una consulta agregada simple, por ejemplo promedio, suma o conteo por categoría
SELECT coin, count(*) as total, max(price) as precio_maximo
FROM workspace.bigdata.coin_data_market
GROUP by coin

coin,total,precio_maximo
bitcoin,372,124739.81088811433
ethereum,372,4817.762321544151
solana,372,247.59944363004428
cardano,372,0.9630994079737264


##Paso 7. Visualización inicial

Genera al menos una visualización sencilla a partir de los datos consultados. Puede ser una gráfica de barras, una tabla resumida o una representación básica de tendencias. El objetivo es reconocer cómo una plataforma de datos también facilita una primera capa de análisis visual.

In [0]:
%sql
--Tendencia de Registros por mes
SELECT date_format(date, 'yyyyMM') AS YYYYMM,
       COUNT(*) AS total
FROM workspace.bigdata.coin_data_market
GROUP BY date_format(date, 'yyyyMM')
ORDER BY YYYYMM;


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5279690095298760>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "--Tendencia de Registros por mes\nSELECT date_format(date, 'yyyyMM') AS YYYYMM,\n       COUNT(*) AS total\nFROM workspace.bigdata.coin_data_market\nGROUP BY date_format(date, 'yyyyMM')\nORDER BY YYYYMM, coin;\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /data

In [0]:
--Precio maximo del bitcon por mes
SELECT date_format(date, 'yyyyMM') AS YYYYMM,
       max(price) AS precio_maximo
FROM workspace.bigdata.coin_data_market
WHERE coin = 'bitcoin'
GROUP BY date_format(date, 'yyyyMM')
ORDER BY YYYYMM;

YYYYMM,precio_maximo
202508,123419.33444694606
202509,117169.11793707692
202510,124739.81088811433
202511,110499.52238345638
202512,93428.59451478458
202601,96898.76071081961
202602,78670.58922752787
202603,74679.37090810013
202604,78637.74356150378
202605,82018.3711243874


Databricks visualization. Run in Databricks to view.

In [0]:

--Visualizacion de coin sin bitcoin y excluyendo registros despues de junio de 2026
SELECT date, price, coin
FROM workspace.bigdata.coin_data_market
WHERE coin NOT IN ('bitcoin') AND date < '2026-06-01'
ORDER BY coin

date,price,coin
2025-11-10,0.5783201871644684,cardano
2026-05-31,0.23581508951182872,cardano
2025-09-03,0.8353364674634308,cardano
2025-11-11,0.59232408864466,cardano
2026-01-08,0.4020882127968396,cardano
2026-02-17,0.30785509644172687,cardano
2026-05-08,0.26281100310116884,cardano
2025-09-04,0.8367687306953374,cardano
2025-09-29,0.8081860522843538,cardano
2025-11-05,0.5213933908892786,cardano


Databricks visualization. Run in Databricks to view.